# Abstract Base Class 

BaseModel is an abstract class. Every model (LSTM, GRU, TFT, etc.) will subclass it and only needs to implement preprocess() and train(). Everything else (data loading, caching, evaluation and saving) is inherited. This avoids copy-pasting the same logic 6 times.

Data split logic:

+ Main dataset: sales_train_evaluation.csv (1941 days of labels in total)
+ Train: d_1-d_1773
+ Val: d_1774-d_1885 (112 days = 3×28 context + 1×28 prediction target)
+ Test: d_1886–d_1941  (56 days = 1×28 context + 1×28 prediction target)
+ This allows models to compute lag features (up to 28 day lag) using the context window in val/test, without data leakage between the splits.

This notebook breaks down BaseModel step by step for learning purposes. The final class is in base_model.py

In [1]:
# Limit of 3 additional packages
# !pip install numpy pandas lightgbm

In [2]:
import os
import random
import pickle
import json
import torch

from abc import ABC, abstractmethod
import numpy as np
import pandas as pd


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'}")

Device: CPU


# 1. Class skeleton

Abstract methods force subclasses to implement specific methods. Every model must have a model_name(), preprocess() and train().

In [ ]:
# Define class skeleton with abstract methods. Document what preprocess and train must do in each subclass

class BaseModel(ABC):

    QUANTILES   = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975]
    PRED_LENGTH = 28
    SEED        = 25
    TARGET_START = 1914

    def __init__(self, data_dir="data", output_dir="outputs"):
        random.seed(self.SEED)
        np.random.seed(self.SEED)
        torch.manual_seed(self.SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.SEED)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        self.device     = DEVICE
        self.data_dir   = data_dir
        self.output_dir = output_dir
        os.makedirs(data_dir,   exist_ok=True)
        os.makedirs(output_dir, exist_ok=True)

        self.train_raw = self.val_raw = self.test_raw = None
        self.calendar  = self.prices  = self.item_weights = None
        self.train_processed = self.val_processed = self.test_processed = None
        self.model = None

    @property
    @abstractmethod
    def model_name(self) -> str: ...

    @abstractmethod
    def preprocess(self): ...
    # Input:  self.train_raw, self.val_raw, self.test_raw
    # Output: self.train_processed, self.val_processed, self.test_processed
    #         (format is model-specific: tensors, lgb.Dataset, DataFrames, etc.)

    @abstractmethod
    def train(self): ...
    # Input:  self.train_processed, self.val_processed
    # Must:
    #   1. Set self.model
    #   2. Save weights → output_dir/{model_name}.pth (or .pkl for LGBM)

    @abstractmethod
    def predict(self): ...
    # Input: self.test_processed, self.output_dir
    # Must: 
    # 1. Load model from: output_dir/{model_name}.pth/pkl 
    # 2. Only use d_1886-d_1913 as context (if needed)
    # 3. Predict 9 quantiles for d_1914-d_1941 in preds_df: a DataFrame (30490 * 28 rows) with columns:
    #     id | day_ahead (1-28) | q0.025 | q0.05 | q0.1 | q0.25 | q0.5 | q0.75 | q0.9 | q0.95 | q0.975
    # 4. Sort predictions by id, day_ahead
    # 5. Make sure quantiles are non-decreasing and >= 0
    # 6. Save predictions → output_dir/{model_name}_predictions.csv
    # Output: preds_df

# Shared methods

    def load_and_split_data(self): pass
    # See section 2: data processing

    def evaluate(self):           pass
    # See section 4: evaluation

    def run_training_pipeline(self):
    # Combines loading and splitting data, preprocessing, and training into a single pipeline
        self.load_and_split_data()
        self.preprocess()
        self.train()

    def run_inference_pipeline(self):
    # Combines inference and evaluation into a single pipeline
        preds_df = self.predict()
        print("Finished model inference.")
        results = self.evaluate(preds_df)
        print("Finished model evaluation.")
        return results

In [4]:
# Sanity check
try:
    b = BaseModel()
except TypeError as e:
    print(e)

Can't instantiate abstract class BaseModel without an implementation for abstract methods 'model_name', 'predict', 'preprocess', 'train'


# 2. Data processing

Changes and notes from Luke's data pipeline (2/4):
1. Do not average price per item and store. When merging sales with prices, use the exact daily price of every item.
2. Melt merged dataframe so that each row only has one item and one day
3. Merge calendar.csv also and set the right data types
4. Some items have missing values for sell_price (they were not on sale for the whole time period). In these cases, forward fill the last available price (or 0 if none available). Then create a new column called 'is_available' so that models take into account out of stock / new / retired items.
5. Check that there are no missing values in the merged dataframe
6. Revenue weight for each item should be calculated as follows: multiply **daily** sales * price, then sum the total revenue for each item during the **last 28 training days only** (as per Kaggle requirement), then normalise so that the total item weights sum to 1. 
7. Categorical encoding issue: assigning each categorical id (store_id, state_id etc) an integer implies a fake distance between them, which will cause bias in embeddings and tree models. Skip this in the base class, but handle it in subclasses - see section 3. How to create your own model subclass

In [5]:
# Step 1: load data
data_dir = "data"
cache = os.path.join(data_dir, "raw_split.pkl")

if os.path.exists(cache):
    print("Loading cached data split...")
    with open(cache, "rb") as f:
        d = pickle.load(f)
else:
    print("Downloading M5 data...")
    base     = "https://huggingface.co/datasets/kashif/M5/resolve/main"
    sales    = pd.read_csv(f"{base}/sales_train_evaluation.csv")
    calendar = pd.read_csv(f"{base}/calendar.csv")
    prices   = pd.read_csv(f"{base}/sell_prices.csv")

    print(sales.shape, calendar.shape, prices.shape)
# (30490, 1947)  (1969, 14)  (6841121, 4)

Loading cached data split...


In [ ]:
# # uncomment if you want to reload data after saving cache
# base     = "https://huggingface.co/datasets/kashif/M5/resolve/main"
# sales    = pd.read_csv(f"{base}/sales_train_evaluation.csv")
# calendar = pd.read_csv(f"{base}/calendar.csv")
# prices   = pd.read_csv(f"{base}/sell_prices.csv")

In [9]:
# Step 2: melt sales to long format
id_cols  = [c for c in sales.columns if not c.startswith("d_")]
day_cols = [c for c in sales.columns if c.startswith("d_")]

sales_long = sales[id_cols + day_cols].melt(
    id_vars=id_cols, var_name='d', value_name='sales'
)
sales_long['d_num'] = sales_long['d'].str[2:].astype(int)

print(sales_long.shape)        # (59,132,490, 9)
print(sales_long.head(3))

(59181090, 9)
                              id        item_id    dept_id   cat_id store_id  \
0  HOBBIES_1_001_CA_1_evaluation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
1  HOBBIES_1_002_CA_1_evaluation  HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1   
2  HOBBIES_1_003_CA_1_evaluation  HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1   

  state_id    d  sales  d_num  
0       CA  d_1      0      1  
1       CA  d_1      0      1  
2       CA  d_1      0      1  


In [10]:
# Step 3: merge calendar data and set dtypes
for col in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
    calendar[col] = calendar[col].fillna('none').astype('category')
calendar['weekday'] = calendar['weekday'].astype('category')
calendar['date']    = pd.to_datetime(calendar['date'])
for col in ['wday', 'month', 'year', 'snap_CA', 'snap_TX', 'snap_WI']:
    calendar[col] = calendar[col].astype('int8')

sales_long = sales_long.merge(calendar, on='d', how='left')

print(sales_long.dtypes)
display(sales_long.sample(5))

id                         str
item_id                    str
dept_id                    str
cat_id                     str
store_id                   str
state_id                   str
d                          str
sales                    int64
d_num                    int64
date            datetime64[us]
wm_yr_wk                 int64
weekday               category
wday                      int8
month                     int8
year                      int8
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap_CA                   int8
snap_TX                   int8
snap_WI                   int8
dtype: object


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,d_num,date,...,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
55679321,HOUSEHOLD_2_437_CA_2_evaluation,HOUSEHOLD_2_437,HOUSEHOLD_2,HOUSEHOLD,CA_2,CA,d_1827,3,1827,2016-01-29,...,7,1,-32,none,none,none,none,0,0,0
44324055,HOUSEHOLD_1_182_WI_1_evaluation,HOUSEHOLD_1_182,HOUSEHOLD_1,HOUSEHOLD,WI_1,WI,d_1454,1,1454,2015-01-21,...,5,1,-33,none,none,none,none,0,0,0
16281494,FOODS_3_659_WI_3_evaluation,FOODS_3_659,FOODS_3,FOODS,WI_3,WI,d_534,5,534,2012-07-15,...,2,7,-36,none,none,none,none,0,1,1
19804761,HOUSEHOLD_2_411_TX_2_evaluation,HOUSEHOLD_2_411,HOUSEHOLD_2,HOUSEHOLD,TX_2,TX,d_650,0,650,2012-11-08,...,6,11,-36,none,none,none,none,1,0,1
22390167,HOUSEHOLD_2_265_CA_4_evaluation,HOUSEHOLD_2_265,HOUSEHOLD_2,HOUSEHOLD,CA_4,CA,d_735,0,735,2013-02-01,...,7,2,-35,none,none,none,none,1,1,0


In [11]:
# Step 4: merge daily prices
sales_long = sales_long.merge(
    prices[['store_id', 'item_id', 'wm_yr_wk', 'sell_price']],  # prices are actually set weekly
    on=['store_id', 'item_id', 'wm_yr_wk'],
    how='left'
)

display(sales_long[['id', 'd', 'wm_yr_wk', 'sell_price']].sample(5))

,id,d,wm_yr_wk,sell_price
33290774,FOODS_1_184_WI_2_evaluation,d_1092,11352,1.00
47838898,HOBBIES_1_091_CA_1_evaluation,d_1570,11516,0.70
41054101,FOODS_3_141_TX_1_evaluation,d_1347,11436,2.88
32430083,HOUSEHOLD_1_362_TX_3_evaluation,d_1064,11348,7.97
13219085,FOODS_1_060_TX_2_evaluation,d_434,11210,NaN


In [12]:
# Step 5: add is_available flag and forward-fill missing prices
sales_long['is_available'] = sales_long['sell_price'].notna().astype('int8')

sales_long = sales_long.sort_values(['id', 'd_num']).reset_index(drop=True)

sales_long['sell_price'] = (
    sales_long.groupby('id')['sell_price']
    .transform(lambda x: x.ffill().fillna(0.0))  
)

print(sales_long.groupby('is_available')['sell_price'].describe())


                   count      mean       std   min   25%   50%   75%     max
is_available                                                                
0             12299413.0  0.000000  0.000000  0.00  0.00  0.00  0.00    0.00
1             46881677.0  4.409438  3.406106  0.01  2.18  3.47  5.84  107.32


Notes:
- is_available=0 rows have sell_price=0 (pre-launch) or ffilled price (temporary pause or post-retirement)
- The output above tells us that all unavailable items are pre-launch since they have no last known price

In [13]:
# Step 6: check missing values
missing = sales_long.isnull().sum()
missing = missing[missing > 0]
assert len(missing) == 0, f"Missing values found:\n{missing}"
print("No missing values. Shape:", sales_long.shape)

No missing values. Shape: (59181090, 24)


In [14]:
# Step 7: split into train/val/test
PRED_LENGTH = 28
total_days = len(day_cols)                        # 1941
train_end  = total_days - 6 * PRED_LENGTH         # 1773
val_end    = total_days - 2 * PRED_LENGTH         # 1885

train_raw = sales_long[sales_long['d_num'] <= train_end].reset_index(drop=True)
val_raw   = sales_long[(sales_long['d_num'] > train_end) & (sales_long['d_num'] <= val_end)].reset_index(drop=True)
test_raw  = sales_long[sales_long['d_num'] > val_end].reset_index(drop=True)

assert train_raw['d_num'].nunique() == 1773
assert val_raw['d_num'].nunique()   == 112
assert test_raw['d_num'].nunique()  == 56
print(f"Train: d_1–d_{train_end} | Val: d_{train_end+1}-d_{val_end} | Test: d_{val_end+1}-d_{total_days}")

Train: d_1–d_1773 | Val: d_1774-d_1885 | Test: d_1886-d_1941


In [15]:
# Step 8: calculate revenue weights on last 28 training days only
last28 = train_raw[train_raw['d_num'] > train_end - 28]
train_rev = (last28['sales'] * last28['sell_price']).groupby(last28['id']).sum()
item_weights = (train_rev / train_rev.sum()).rename('weight')

print(f"Weights sum to: {item_weights.sum():.6f}")    # should be 1.0
print(f"Non-zero items: {(item_weights > 0).sum()} / {len(item_weights)}")
print(item_weights.sort_values(ascending=False).head(5))

Weights sum to: 1.000000
Non-zero items: 27040 / 30490
id
HOBBIES_1_354_TX_3_evaluation    0.004617
HOBBIES_1_158_TX_3_evaluation    0.004371
FOODS_3_120_CA_3_evaluation      0.003082
FOODS_1_096_WI_2_evaluation      0.002459
HOBBIES_1_354_TX_2_evaluation    0.002038
Name: weight, dtype: float64


In [16]:
# sense check
print([train_raw.id.nunique(), train_rev.shape, item_weights.shape])

[30490, (30490,), (30490,)]


## 2.1 Full load_and_split_data function

In [17]:
# Put it all together in one function. Run once and cache to disk. Every subclass reuses the same data split: train 1773 days, val 112 days, test 56 days

def load_and_split_data(self):
    """
    Downloads (or loads from cache) M5 data and processes it into long-format dataframes (1 row per item and day). 
    Split into train (d_1-d_1773), val (d_1774-d_1885), and test (d_1886-d_1941).
    Save raw splits to cache for loading when training models.

    Output: self.train_raw, self.val_raw, self.test_raw, self.item_weights
    """
    # Step 1: load data
    cache = os.path.join(self.data_dir, "raw_split.pkl")

    if os.path.exists(cache):
        with open(cache, "rb") as f:
            d = pickle.load(f)
        print("Loaded cached data splits.")
    else:
        base     = "https://huggingface.co/datasets/kashif/M5/resolve/main"
        sales    = pd.read_csv(f"{base}/sales_train_evaluation.csv")
        calendar = pd.read_csv(f"{base}/calendar.csv")
        prices   = pd.read_csv(f"{base}/sell_prices.csv")
        print("Downloaded M5 data.")

        # Step 2: melt sales to long format
        id_cols  = [c for c in sales.columns if not c.startswith("d_")]
        day_cols = [c for c in sales.columns if c.startswith("d_")]
        sales_long = sales[id_cols + day_cols].melt(
            id_vars=id_cols, var_name='d', value_name='sales'
        )
        sales_long['d_num'] = sales_long['d'].str[2:].astype(int)

        # Step 3: merge calendar data and set dtypes
        for col in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
            calendar[col] = calendar[col].fillna('none').astype('category')
        calendar['weekday'] = calendar['weekday'].astype('category')
        calendar['date']    = pd.to_datetime(calendar['date'])
        for col in ['wday', 'month', 'year', 'snap_CA', 'snap_TX', 'snap_WI']:
            calendar[col] = calendar[col].astype('int8')
        sales_long = sales_long.merge(calendar, on='d', how='left')

        # Step 4: merge daily prices
        sales_long = sales_long.merge(
            prices[['store_id', 'item_id', 'wm_yr_wk', 'sell_price']],
            on=['store_id', 'item_id', 'wm_yr_wk'], how='left'
        )

        # Step 5: add is_available flag and forward-fill missing prices
        sales_long['is_available'] = sales_long['sell_price'].notna().astype('int8')
        sales_long = sales_long.sort_values(['id', 'd_num']).reset_index(drop=True)
        sales_long['sell_price'] = (
            sales_long.groupby('id')['sell_price']
            .transform(lambda x: x.ffill().fillna(0.0))
        )

        # Step 6: check missing values
        missing = sales_long.isnull().sum()
        missing = missing[missing > 0]
        assert len(missing) == 0, f"Missing values found:\n{missing}"

        # Step 7: split into train/val/test
        total_days = len(day_cols)
        train_end  = total_days - 6 * self.PRED_LENGTH   # 1773
        val_end    = total_days - 2 * self.PRED_LENGTH   # 1885

        train_raw = sales_long[sales_long['d_num'] <= train_end].reset_index(drop=True)
        val_raw   = sales_long[(sales_long['d_num'] > train_end) & (sales_long['d_num'] <= val_end)].reset_index(drop=True)
        test_raw  = sales_long[sales_long['d_num'] > val_end].reset_index(drop=True)

        assert train_raw['d_num'].nunique() == 1773
        assert val_raw['d_num'].nunique()   == 112
        assert test_raw['d_num'].nunique()  == 56
        print(f"Train: d_1-d_{train_end} | Val: d_{train_end+1}-d_{val_end} | Test: d_{val_end+1}-d_{total_days}")

        # Step 8: calculate revenue weights on last 28 training days only
        last28 = train_raw[train_raw['d_num'] > train_end - 28]
        train_rev = (last28['sales'] * last28['sell_price']).groupby(last28['id']).sum()
        item_weights = (train_rev / train_rev.sum()).rename('weight')

        # Step 9: save data splits to cache
        d = dict(train_raw=train_raw, val_raw=val_raw, test_raw=test_raw,
                    item_weights=item_weights)

        # save all splits and weights into raw_split.pkl
        with open(cache, "wb") as f:
            pickle.dump(d, f)
        print(f"Cached to {cache}")

    self.train_raw    = d["train_raw"]
    self.val_raw      = d["val_raw"]
    self.test_raw     = d["test_raw"]
    self.item_weights = d["item_weights"]
    print("Finished data processing.")

    return self.train_raw, self.val_raw, self.test_raw, self.item_weights

BaseModel.load_and_split_data = load_and_split_data

In [ ]:
# Test the function

class ConcreteModel(BaseModel):
    """Minimal concrete subclass so we can instantiate BaseModel for testing."""
    @property
    def model_name(self): return "test_model"
    def preprocess(self): pass
    def train(self):      pass
    def predict(self):      pass


def test_load_and_data_split():
    model = ConcreteModel(data_dir="data")

    train, val, test, weights = model.load_and_split_data()

    # Test 1: correct number of unique days in each split
    assert train['d_num'].nunique() == 1773, "Train should have 1773 days"
    assert val['d_num'].nunique()   == 112,  "Val should have 112 days"
    assert test['d_num'].nunique()  == 56,   "Test should have 56 days"
    print("Test 1 passed: split sizes correct")

    # Test 2: no temporal overlap between splits
    assert train['d_num'].max() < val['d_num'].min(), "Train and val overlap"
    assert val['d_num'].max()   < test['d_num'].min(), "Val and test overlap"
    print("Test 2 passed: no temporal overlap")

    # Test 3: no missing values in any split
    for name, df in [("train", train), ("val", val), ("test", test)]:
        assert df.isnull().sum().sum() == 0, f"Missing values in {name}"
    print("Test 3 passed: no missing values")

    # Test 4: is_available is binary and sell_price is non-negative
    for name, df in [("train", train), ("val", val), ("test", test)]:
        assert df['is_available'].isin([0, 1]).all(), f"is_available not binary in {name}"
        assert (df['sell_price'] >= 0).all(), f"Negative sell_price in {name}"
    print("Test 4 passed: is_available binary, sell_price non-negative")

    # Test 5: item_weights sum to 1 and are all non-negative
    assert abs(model.item_weights.sum() - 1.0) < 1e-6, "Weights do not sum to 1"
    assert (model.item_weights >= 0).all(), "Negative weights found"
    print("Test 5 passed: item_weights valid")

    print("\nAll tests passed.")


test_load_and_data_split()

Loaded cached data splits.
Finished data processing.
Loaded cached data splits.
Finished data processing.
Test 1 passed: split sizes correct
Test 2 passed: no temporal overlap
Test 3 passed: no missing values
Test 4 passed: is_available binary, sell_price non-negative
Test 5 passed: item_weights valid

All tests passed.


# 3. How to create your own model subclass

Suggestions (Yen): 
1. LSTM, GRU, Hierarchical LSTM should have consistent preprocess methods - check how you are encoding categorical variables and creating embeddings and lags.
2. For LGBM / LGBM+NN, use lgb.Dataset and set categorical_feature = [list of cat_features]
3. For TFT, just use raw string categories in TimeSeriesDataset without encoding

In [20]:
# Make sure you have base_model.py in your working directory. Import the following packages and BaseModel class

import json
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
from base_model import BaseModel

In [21]:
# Define your model class by inheriting from BaseModel. 
# Set model_name and define the methods preprocess, train and predict

class LinearModel(BaseModel):

    model_name = "linear"

    def preprocess(self):
        # Use a small fixed sample for speed - this is a demo, not a real model
        for attr, df in [('train_processed', self.train_raw),
                         ('val_processed',   self.val_raw),
                         ('test_processed',  self.test_raw)]:
            sample = df.sample(n=min(10_000, len(df)), random_state=self.SEED)
            X = torch.tensor(
                sample[['wday', 'month', 'sell_price', 'is_available']].values,
                dtype=torch.float32
            )
            y = torch.tensor(sample['sales'].values, dtype=torch.float32).unsqueeze(1)
            setattr(self, attr, (X, y))

    def train(self, epochs=10, lr=1e-3):
        X_train, y_train = self.train_processed
        X_val,   y_val   = self.val_processed

        self.model = nn.Linear(X_train.shape[1], 1).to(self.device)
        optimiser  = torch.optim.Adam(self.model.parameters(), lr=lr)
        loss_fn    = nn.MSELoss()

        for epoch in range(epochs):
            self.model.train()
            optimiser.zero_grad()
            loss_fn(self.model(X_train.to(self.device)), y_train.to(self.device)).backward()
            optimiser.step()

            with torch.no_grad():
                val_loss = loss_fn(self.model(X_val.to(self.device)), y_val.to(self.device)).item()
            print(f"Epoch {epoch+1:02d}/{epochs} | val MSE: {val_loss:.4f}")

        torch.save(self.model.state_dict(), f"{self.output_dir}/{self.model_name}.pth")
    
    def predict(self) -> pd.DataFrame:
        # 1. Load weights
        weights_path = os.path.join(self.output_dir, f"{self.model_name}.pth")
        self.model = nn.Linear(4, 1).to(self.device)
        self.model.load_state_dict(torch.load(weights_path, map_location=self.device))
        self.model.eval()

        # 2. Build features from context window (d_1886–d_1913) is not needed
        #    for this model — it uses same static features for all days.
        #    Features are built from target window (d_1914–d_1941) directly.
        target = self.test_raw[self.test_raw['d_num'] >= self.TARGET_START].copy()
        X = torch.tensor(
            target[['wday', 'month', 'sell_price', 'is_available']].values,
            dtype=torch.float32
        )

        # 3. Run inference
        with torch.no_grad():
            point_preds = self.model(X.to(self.device)).cpu().numpy().flatten()

        # 4. Build DataFrame
        q_cols   = [f"q{q}" for q in self.QUANTILES]
        preds_df = target[['id', 'd_num']].copy().reset_index(drop=True)
        preds_df['day_ahead'] = preds_df['d_num'] - self.TARGET_START + 1
        for col in q_cols:
            preds_df[col] = point_preds

        # 5. Enforce non-decreasing quantiles >= 0
        preds_df[q_cols] = (
            preds_df[q_cols]
            .clip(lower=0)
            .apply(lambda row: np.maximum.accumulate(row.values), axis=1, result_type='expand')
            .set_axis(q_cols, axis=1)
        )

        # 6. Sort and save
        preds_df = (
            preds_df[['id', 'day_ahead'] + q_cols]
            .sort_values(['id', 'day_ahead'])
            .reset_index(drop=True)
        )
        out_path = os.path.join(self.output_dir, f"{self.model_name}_predictions.csv")
        preds_df.to_csv(out_path, index=False)
        print(f"Predictions saved to {out_path}")

        return preds_df

In [22]:
# Test your class - make sure preprocess, train, and predict run without errors
m = LinearModel()
m.load_and_split_data()  # takes about 4mins
m.preprocess()
m.train() 
preds_df = m.predict()
print(preds_df.shape) # should be (853720, 11)

Loaded cached data splits.
Finished data processing.
Epoch 01/10 | val MSE: 24.7562
Epoch 02/10 | val MSE: 24.6459
Epoch 03/10 | val MSE: 24.5361
Epoch 04/10 | val MSE: 24.4269
Epoch 05/10 | val MSE: 24.3182
Epoch 06/10 | val MSE: 24.2101
Epoch 07/10 | val MSE: 24.1025
Epoch 08/10 | val MSE: 23.9954
Epoch 09/10 | val MSE: 23.8890
Epoch 10/10 | val MSE: 23.7831
Predictions saved to outputs\linear_predictions.csv
(853720, 11)


# 4. Evaluation (WIP)

The inference pipeline is designed to be run independently after the training pipeline.
- predict() is model-specific. It loads test_processed from cache, runs models on test data, and saves predictions as CSV
- evaluate() loads the saved predictions, computes evaluation metrics and saves the results.

In [23]:
# Step 1: extract test targets
TARGET_START = 1914  # d_1914 and d_1941 (predict last 28 days)

targets = (
    test_raw[test_raw['d_num'] >= TARGET_START]
    .pivot(index='id', columns='d_num', values='sales')
    .sort_index()
)
print(targets.shape)         # (30490, 28)
print(targets.iloc[:3, :5])  # sanity check: first 3 series, first 5 days

(30490, 28)
d_num                        1914  1915  1916  1917  1918
id                                                       
FOODS_1_001_CA_1_evaluation     2     0     0     0     0
FOODS_1_001_CA_2_evaluation     0     3     0     0     0
FOODS_1_001_CA_3_evaluation     1     0     1     0     8


In the winning paper (Eq. 5), we scale pinball loss by the in-sample mean absolute day-to-day change. This penalises errors less if the series (sales pattern for that item) is more volatile. 
- clip(lower=1e-8) prevents 0 or inf WSPL from a single bad prediction from an item with zero sales during the training window

In [24]:
# Step 2: calculate scale
train_sorted = train_raw.sort_values(['id', 'd_num']).copy()
train_sorted['prev_sales'] = train_sorted.groupby('id')['sales'].shift(1)

scale = (
    train_sorted.dropna(subset=['prev_sales'])
    .assign(abs_diff=lambda df: (df['sales'] - df['prev_sales']).abs())
    .groupby('id')['abs_diff']
    .mean()
    .clip(lower=1e-8)  # avoid division by zero for flat/zero series
)
print(scale.describe())
print(f"Zero-clipped series: {(scale <= 1e-8).sum()}")

count    3.049000e+04
mean     9.074526e-01
std      1.289095e+00
min      1.000000e-08
25%      2.550790e-01
50%      5.395034e-01
75%      1.054176e+00
max      4.194018e+01
Name: abs_diff, dtype: float64
Zero-clipped series: 27


In [26]:
# Step 3: calculate WSPL - assums that preds_df already exists with correct format and values

QUANTILES = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975]

q_cols = [f"q{q}" for q in QUANTILES]
ids    = targets.index.tolist()

# Pivot to (N, 28, 9) — one shot instead of 30490 DataFrame scans
preds_pivot = (
    preds_df.set_index(['id', 'day_ahead'])[q_cols]
    .unstack('day_ahead')        # (N, 9*28) MultiIndex columns
    .reindex(ids)                # align row order with targets
)
q_arr  = preds_pivot.values.reshape(len(ids), len(q_cols), 28)  # (N, 9, 28)
y_mat  = targets.values.astype(np.float32)                       # (N, 28)
q_vals = np.array(QUANTILES)                                # (9,)

# Pinball loss: (N, 9, 28)
errors  = y_mat[:, None, :] - q_arr                              # (N, 9, 28)
pinball = np.maximum(q_vals[None, :, None] * errors,
                     (q_vals[None, :, None] - 1) * errors)

# Mean over days and quantiles → (N,), then scale
wspl_per_series = pinball.mean(axis=(1, 2)) / scale.reindex(ids).values
weights = item_weights.reindex(ids).fillna(0).values
wspl    = float(np.dot(weights, wspl_per_series))

## 3.1 Full evaluate function

In [ ]:
def evaluate(self) -> dict:
    """
    Load saved test_processed and model weights, run predict(), compute WSPL
    on d_1914–d_1941, and save results to output_dir/{model_name}_results.json.
    """
    # 1. Load raw splits
    cache = os.path.join(self.data_dir, "raw_split.pkl")
    assert os.path.exists(cache), "Run load_and_split_data() first."
    with open(cache, "rb") as f:
        d = pickle.load(f)
    self.train_raw    = d["train_raw"]
    self.test_raw     = d["test_raw"]
    self.item_weights = d["item_weights"]

    # 2. Load test_processed
    proc_path = os.path.join(self.output_dir, f"{self.model_name}_test_processed.pkl")
    assert os.path.exists(proc_path), f"No test_processed at {proc_path}. Run run_training_pipeline() first."
    with open(proc_path, "rb") as f:
        self.test_processed = pickle.load(f)

    # 3. Run inference and save predictions CSV
    preds_df = self.predict()

    # 4. Extract test targets: d_1914–d_1941
    targets = (
        self.test_raw[self.test_raw['d_num'] >= self.TARGET_START]
        .pivot(index='id', columns='d_num', values='sales')
        .sort_index()
    )

    # 5. Naive MAE scale: mean |y_t - y_{t-1}| over training window
    train_s        = self.train_raw.sort_values(['id', 'd_num']).copy()
    train_s['prev'] = train_s.groupby('id')['sales'].shift(1)
    scale = (
        train_s.dropna(subset=['prev'])
        .assign(abs_diff=lambda df: (df['sales'] - df['prev']).abs())
        .groupby('id')['abs_diff']
        .mean()
        .clip(lower=1e-8)
    )

    # 6. Vectorised WSPL
    q_cols = [f"q{q}" for q in self.QUANTILES]
    ids    = targets.index.tolist()

    q_arr  = (
        preds_df.set_index(['id', 'day_ahead'])[q_cols]
        .unstack('day_ahead')
        .reindex(ids)
        .values.reshape(len(ids), len(q_cols), self.PRED_LENGTH)
    )                                                               # (N, 9, 28)
    y_mat  = targets.values.astype(np.float32)                     # (N, 28)

## Linear example - inference

In [ ]:
m.evaluate()

In [ ]:
# Finally, orchestrate the full training pipeline:
m.run_training_pipeline()

# and then run the inference pipeline:
m.run_inference_pipeline()

TODO:

- Define a separate function that takes the raw splits, filters for selected stores and filters for the top num_items by highest total sales in the last num_days (calculated in the training split only). the arguments of the function are: store_ids (list, default: ['CA_3']), num_items (default: 100), last_num_days (default: 365)

# 5. Notes on MLOps best practice

When working in industrial AI teams, each model should output one folder outputs/{model_name}/, which saves:
- model.pth - learned weights or fittied model object
- preprocess.pkl - output of preprocess() i.e. data splits used during training
- config.json - model hyperparameters and other preprocessing settings used e.g. hidden size, seq length, features, prediction horizon
- metrics.json - training and validation metrics

This makes it very easy for other team members to reproduce the results of your model run, pick things up when pipelines fail halfway, and save a lot of time during code review.